In [ ]:
pip install scikit-learn pandas numpy

In [127]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

class GameRecommender:
    
    def __init__(self, db_path):
        self.db_path = db_path
        self.user_id = None
        self.vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)
        self.tfidf_matrix = None
        self.games_df = None
        self.genres_list = None
        self._load_data()
    
    def _get_connection(self):
        conn = sqlite3.connect(self.db_path)
        conn.row_factory = sqlite3.Row
        return conn
    
    def _load_data(self):

        conn = self._get_connection()
        cursor = conn.cursor()

        cursor.execute('''
            SELECT 
                vg.id,
                vg.title,
                gtp.insults_flaming,
                gtp.hate_harassment,
                gtp.offensive_texts
            FROM video_games vg
            LEFT JOIN game_toxicity_prevalence_profile gtp 
                ON vg.id = gtp.game_id
        ''')

        games = cursor.fetchall()

        cursor.execute('SELECT id, name FROM genres')
        genre_rows = cursor.fetchall()
        self.genres_list = [row['name'] for row in genre_rows]

        cursor.execute('''
            SELECT gg.game_id, g.name as genre_name
            FROM game_genres gg
            JOIN genres g ON gg.genre_id = g.id
        ''')
        
        game_genres = cursor.fetchall()

        genre_map = {}
        for row in game_genres:
            game_id = row['game_id']
            if game_id not in genre_map:
                genre_map[game_id] = []
            genre_map[game_id].append(row['genre_name'])
        
        conn.close()
        
        data = []
        for game in games:
            game_id = game['id']
            genres_list = genre_map.get(game_id, [])
            genres_str = ','.join(genres_list)
            
            data.append({
                'id': game['id'],
                'title': game['title'],
                'genres': genres_str,
                'genres_list': genres_list,
                'insults_flaming': float(game['insults_flaming']),
                'hate_harassment': float(game['hate_harassment']),
                'offensive_texts': float(game['offensive_texts'])
            })
        
        self.games_df = pd.DataFrame(data)
        
        self.games_df['combined_features'] = self.games_df['genres'].fillna('')
        self.tfidf_matrix = self.vectorizer.fit_transform(self.games_df['combined_features'])
        
        print(f"Loaded {len(self.games_df)} games from Database")
        print(f"Genres: {len(self.get_all_genres())} available")
    
    def create_user(self, username):

        conn = self._get_connection()
        cursor = conn.cursor()
        username = str(username).strip()

        cursor.execute('SELECT id FROM users WHERE username = ?', (username,))
        user = cursor.fetchone()
        
        if user:
            self.user_id = user['id']
            conn.close()
            return {
                'user_id': self.user_id,
                'username': username,
                'status': 'existing'
            }

        cursor.execute('''
            INSERT INTO users (username, created_at)
            VALUES (?, CURRENT_TIMESTAMP)
        ''', (username,))
        
        user_id = cursor.lastrowid

        cursor.execute('''
            INSERT INTO user_toxicity_tolerance (
                user_id, insults_flaming, hate_harassment, offensive_texts
            ) VALUES (?, 1.0, 1.0, 1.0)
        ''', (user_id,))
        
        conn.commit()
        conn.close()
        
        self.user_id = user_id
        return {
            'user_id': self.user_id,
            'username': username,
            'status': 'new'
        }
    
    def get_all_genres(self):
        return self.genres_list
    
    def get_user_preferences(self):

        if not self.user_id:
            return None
        
        conn = self._get_connection()
        cursor = conn.cursor()
        
        cursor.execute('''
            SELECT g.id, g.name
            FROM user_preferences up
            JOIN genres g ON up.genre_id = g.id
            WHERE up.user_id = ?
        ''', (self.user_id,))
        
        genre_rows = cursor.fetchall()
        preferred_genres = [row['name'] for row in genre_rows]
        
        cursor.execute('''
            SELECT insults_flaming, hate_harassment, offensive_texts
            FROM user_toxicity_tolerance
            WHERE user_id = ?
        ''', (self.user_id,))
        
        tolerance = cursor.fetchone()
        conn.close()
        
        if not tolerance:
            return {
                'genres': preferred_genres,
                'toxicity_tolerance': {
                    'insults_flaming': 1.0,
                    'hate_harassment': 1.0,
                    'offensive_texts': 1.0
                }
            }
        
        return {
            'genres': preferred_genres,
            'toxicity_tolerance': {
                'insults_flaming': float(tolerance['insults_flaming']),
                'hate_harassment': float(tolerance['hate_harassment']),
                'offensive_texts': float(tolerance['offensive_texts'])
            }
        }
    
    def set_preferences(self, genres, toxicity_tolerance):
        if not self.user_id:
            print("Please create a user first!")
            return False
        
        conn = self._get_connection()
        cursor = conn.cursor()
        
        try:
            cursor.execute('DELETE FROM user_preferences WHERE user_id = ?', (self.user_id,))
            
            for genre_name in genres:
                cursor.execute('SELECT id FROM genres WHERE name = ?', (genre_name,))
                genre = cursor.fetchone()
                if genre:
                    cursor.execute('''
                        INSERT INTO user_preferences (user_id, genre_id)
                        VALUES (?, ?)
                    ''', (self.user_id, genre['id']))
            
            cursor.execute('''
                UPDATE user_toxicity_tolerance 
                SET 
                    insults_flaming = ?,
                    hate_harassment = ?,
                    offensive_texts = ?
                WHERE user_id = ?
            ''', (
                toxicity_tolerance.get('insults_flaming'),
                toxicity_tolerance.get('hate_harassment'),
                toxicity_tolerance.get('offensive_texts'),
                self.user_id
            ))
            
            conn.commit()
            conn.close()
            return True
            
        except Exception as e:
            conn.rollback()
            conn.close()
            print(f"Error setting preferences: {e}")
            return False
    
    def check_toxicity_compatibility(self, game_row, tolerance):
        violations = {}
        is_compatible = True
        
        for dim in ['insults_flaming', 'hate_harassment', 'offensive_texts']:
            game_val = game_row[dim]
            tol_val = tolerance[dim]
            
            if game_val > tol_val:
                violations[dim] = game_val - tol_val
                is_compatible = False
            else:
                violations[dim] = 0
        
        return is_compatible, violations
    
    def calculate_hinge_violation(self, game_row, tolerance):
        violations = []
        
        for dim in ['insults_flaming', 'hate_harassment', 'offensive_texts']:
            game_val = game_row[dim]
            tol_val = tolerance[dim]
            violation = max(0, game_val - tol_val)
            violations.append(violation)
        
        avg_violation = np.mean(violations)
        max_violation = max(violations)
        
        return {
            'avg_violation': round(avg_violation, 4),
            'max_violation': round(max_violation, 4),
            'violations': {
                'insults_flaming': round(violations[0], 4),
                'hate_harassment': round(violations[1], 4),
                'offensive_texts': round(violations[2], 4)
            }
        }
    
    def calculate_toxicity_compatibility_score(self, game_row, tolerance):
        
        epsilon = 1e-10
        total_violation = 0
        
        for dim in ['insults_flaming', 'hate_harassment', 'offensive_texts']:
            game_val = game_row[dim]
            tol_val = tolerance[dim]
            
            violation = max(0, game_val - tol_val)
            max_possible = max(epsilon, 1 - tol_val)
            normalized_violation = violation / max_possible
            total_violation += normalized_violation
        
        avg_violation = total_violation / 3
        compatibility_score = 1 - avg_violation
        compatibility_score = max(0, compatibility_score)
        
        return round(compatibility_score, 4)
    
    def get_recommendations(self, n_recommendations = 5, mode = 'strict'):
        
        if not self.user_id:
            print("Please create a user first!")
            return None
        
        preferences = self.get_user_preferences()
        
        if len(preferences['genres']) == 0:
            print("Please set your genre preferences first!")
            return None
        
        tolerance = preferences['toxicity_tolerance']
        preferred_genres = set(preferences['genres'])
        
        genre_str = ' '.join(preferences['genres'])
        user_vector = self.vectorizer.transform([genre_str])
        genre_similarities = cosine_similarity(user_vector, self.tfidf_matrix).flatten()
        
        scored_games = []
        
        for idx, row in self.games_df.iterrows():
            game_id = row['id']
            genre_sim = genre_similarities[idx]
            
            if isinstance(row['genres_list'], list):
                game_genres = set(row['genres_list'])
            else:
                game_genres = set(row['genres'].split(',')) if row['genres'] else set()
            
            shared_genres = preferred_genres & game_genres
            has_matching_genre = len(shared_genres) > 0
            match_count = len(shared_genres)
            
            is_compatible, violations = self.check_toxicity_compatibility(row, tolerance)
            
            tox_score = self.calculate_toxicity_compatibility_score(row, tolerance)
            
            if mode == 'strict':
                if not has_matching_genre:
                    continue 
                if not is_compatible:
                    continue
                
                score = np.sqrt(genre_sim * tox_score)
                
            elif mode == 'balanced':
                if not has_matching_genre:
                    continue 
                
                if is_compatible:
                    score = np.sqrt(genre_sim * tox_score)
                else:
                    hinge = self.calculate_hinge_violation(row, tolerance)
                    penalty = hinge['avg_violation'] * 0.5
                    score = np.sqrt(genre_sim * (tox_score - penalty))
                    score = max(0, score)
            
            
            scored_games.append({
                'id': int(game_id),
                'row': row,
                'genre_similarity': round(genre_sim, 4),
                'toxicity_compatibility': tox_score,
                'combined_score': round(score, 4),
                'shared_genres': list(shared_genres),
                'has_matching_genre': has_matching_genre,
                'is_toxicity_compatible': is_compatible,
                'match_count': match_count,
                'violations': violations
            })
        
        scored_games.sort(key=lambda x: x['combined_score'], reverse=True)
        
        recommendations = []
        for game_data in scored_games[:n_recommendations]:
            row = game_data['row']
            recommendations.append({
                'id': game_data['id'],
                'title': row['title'],
                'genres': row['genres'].split(',') if row['genres'] else [],
                'shared_genres': game_data['shared_genres'],
                'match_count': game_data['match_count'],
                'toxicity_prevalence': {
                    'insults_flaming': float(row['insults_flaming']),
                    'hate_harassment': float(row['hate_harassment']),
                    'offensive_texts': float(row['offensive_texts'])
                },
                'toxicity_match': {
                    'is_eligible': game_data['is_toxicity_compatible'],
                    'genre_similarity': game_data['genre_similarity'],
                    'toxicity_compatibility': game_data['toxicity_compatibility'],
                    'combined_score': game_data['combined_score'],
                    'violations': game_data['violations']
                },
                'recommendation_score': game_data['combined_score']
            })
        
        return recommendations

def display_recommendations(recommendations):
    if not recommendations:
        print("No recommendations available")
        return

    toxicity_dict = {
        "insults_flaming": "Insults and Flaming",
        "offensive_texts": "Other Offensive Texts",
        "hate_harassment": "Hate and Harassment"
    }
    
    print(f"\nTop {len(recommendations)} Recommendations")
    print("\n" + "="*80)
    
    for i, game in enumerate(recommendations, 1):
        
        is_eligible = game['toxicity_match']['is_eligible']
        status_text = "Meets all toxicity limits" if is_eligible else "FALLBACK - Exceeds some limits"
        
        print(f"\n{i}. {game['title']} (Score: {game['recommendation_score']:.3f})")
        print(f"   {status_text}")
        print(f"   Genres: {', '.join(game['genres'])}")
        
        if game['shared_genres']:
            print(f"   Matching Genres: {', '.join(game['shared_genres'])} (Matches: {game['match_count']})")
        else:
            print(f"   No matching genres found")
        
        tox = game['toxicity_prevalence']
        print(f"   Toxicity Prevalence:")
        print(f"     • Insults and Flaming: {tox['insults_flaming']*100:.2f}%")
        print(f"     • Other Offensive Text: {tox['offensive_texts']*100:.2f}%")
        print(f"     • Hate and Harassment: {tox['hate_harassment']*100:.2f}%")
        
        match = game['toxicity_match']
        if is_eligible:
            print(f"   Toxicity Match:")
            print(f"     • Genre Similarity Score: {match['genre_similarity']:.2f}")
            print(f"     • Toxicity Compatibility Score: {match['toxicity_compatibility']:.2f}")
        else:
            print(f"   Toxicity Violations (Fallback):")
            print(f"     • Genre Similarity Score: {match['genre_similarity']:.2f}")
            print(f"     • Toxicity Compatibility Score: {match['toxicity_compatibility']:.2f}")
            for dim, val in match['violations'].items():
                if val > 0:
                    print(f"     • {toxicity_dict[dim]}: {val*100:.2f}% above limit")


recommender = GameRecommender('Database/game_recommendation_system.db')

Loaded 15 games from Database
Genres: 18 available


# User Setup

In [35]:
USERNAME = "Derick"

In [86]:
user = recommender.create_user(USERNAME)
status = "Welcome back" if user['status'] == 'existing' else "New user created"
print(f"\n{status}: {user['username']} (ID: {user['user_id']})")


Welcome back: Derick (ID: 1)


# Set/Update Preferences

In [46]:
all_genres = recommender.get_all_genres()
print("\nAvailable Genres:")
for i, genre in enumerate(sorted(all_genres), 1):
    print(f"  {i:2d}. {genre}")


Available Genres:
   1. Action
   2. Adventure
   3. Battle Royale
   4. FPS
   5. Fantasy
   6. Horror
   7. MMORPG
   8. MOBA
   9. Multiplayer
  10. RPG
  11. Sandbox
  12. Sci-Fi
  13. Shooter
  14. Sports
  15. Strategy
  16. Survival
  17. Tactical
  18. Team-based


In [117]:
TOL_INSULTS_AND_FLAMING = 0.15
TOL_HATE_AND_HARASSMENT = 0.01
TOL_OFFENSIVE_TEXTS = 0.20
PREF_GENRE_STR = "FPS, Action, Adventure"

In [119]:
genre_input = PREF_GENRE_STR

preferred_genres = [g.strip() for g in genre_input.split(',') if g.strip()]               

valid_genres = []
for genre in preferred_genres:
    if genre in all_genres:
         valid_genres.append(genre)
    else:
        print(f"Warning: '{genre}' is not a valid genre.")
                
    if not valid_genres:
        print("No valid genres selected!")

toxicity_tolerance = {
                    'insults_flaming': TOL_INSULTS_AND_FLAMING,
                    'hate_harassment': TOL_HATE_AND_HARASSMENT,
                    'offensive_texts': TOL_OFFENSIVE_TEXTS
                }

if recommender.set_preferences(valid_genres, toxicity_tolerance):
    print("\nPreferences updated successfully!")


Preferences updated successfully!


# Get Recommendations

In [125]:
TOP_K = 3
RECOMMENDER_MODE = "strict"

In [126]:
recommendations = recommender.get_recommendations(TOP_K, mode=RECOMMENDER_MODE)
display_recommendations(recommendations)


Top 3 Recommendations


1. Terraria (Score: 0.808)
   Meets all toxicity limits
   Genres: Sandbox, Action, Adventure
   Matching Genres: Action, Adventure (Matches: 2)
   Toxicity Prevalence:
     • Insults and Flaming: 2.44%
     • Other Offensive Text: 2.19%
     • Hate and Harassment: 0.13%
   Toxicity Match:
     • Genre Similarity Score: 0.65
     • Toxicity Compatibility Score: 1.00

2. Minecraft (Score: 0.624)
   Meets all toxicity limits
   Genres: Sandbox, Survival, Adventure
   Matching Genres: Adventure (Matches: 1)
   Toxicity Prevalence:
     • Insults and Flaming: 3.40%
     • Other Offensive Text: 1.89%
     • Hate and Harassment: 0.00%
   Toxicity Match:
     • Genre Similarity Score: 0.39
     • Toxicity Compatibility Score: 1.00

3. Tom Clancys Rainbow Six Siege (Score: 0.542)
   Meets all toxicity limits
   Genres: FPS, Tactical, Multiplayer
   Matching Genres: FPS (Matches: 1)
   Toxicity Prevalence:
     • Insults and Flaming: 9.41%
     • Other Offensive Text: 9